<a href="https://colab.research.google.com/github/hayoung-kwon/study_2026/blob/main/1%EC%9D%BC%EC%B0%A8_%EC%8B%A4%EC%8A%B5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
movie_data = [
    ["Extreme Job", "Comedy", 2, 3, 8.5, 6],
    ["Train to Busan", "Action", 2, 3, 8.0, 7],
    ["Parasite", "Drama", 3, 3, 9.2, 5],
    ["Miracle in Cell No.7", "Drama", 3, 2, 8.4, 4],
    ["The Admiral", "Historical", 4, 2, 8.1, 8],
    ["Along with the Gods", "Fantasy", 2, 3, 8.2, 5],
    ["Oldboy", "Thriller", 3, 1, 8.7, 8],
    ["The Outlaws", "Action", 3, 3, 8.3, 8],
    ["The Roundup", "Action", 3, 4, 8.6, 8],
    ["Veteran", "Action", 3, 3, 8.2, 7],
    ["Assassination", "Action", 3, 3, 8.3, 6],
    ["Ode to My Father", "Drama", 4, 2, 8.5, 5],
    ["A Taxi Driver", "Drama", 3, 3, 8.8, 5],
    ["The Attorney", "Drama", 4, 2, 8.9, 5],
    ["The Thieves", "Crime", 2, 2, 8.1, 6],
    ["New World", "Crime", 3, 2, 8.8, 8],
    ["The Face Reader", "Historical", 4, 2, 8.2, 6],
    ["Frozen", "Animation", 1, 2, 8.0, 3],
    ["Elemental", "Animation", 1, 4, 8.3, 4],
    ["Avatar", "SF", 2, 2, 8.9, 7]
]

import numpy as np
import pandas as pd

#DataFrame으로 변환하기
columns=["Movie","Genre","Age","Release","Rating","Male"]
movie_df=pd.DataFrame(movie_data, columns=columns)

#평점을 높은 순부터 정렬
# sort_values에서 맨 앞 인수는 by=이라 그거 생략하고 "Rating"만 바로 써도 되긴 한다
movie_df.sort_values(by="Rating", ascending=False)

#장르가 Action인것만 모아보기
movie_df[movie_df["Genre"]=="Action"]

# 문자는 원핫인코딩으로 해야하므로 따로 벡터에 들어갈 표 만들기
# get_dummies로 원핫인코딩
genre = pd.get_dummies(movie_df["Genre"])

movie_df[["Age","Release","Rating","Male"]]
#스케일링하기
scaled = movie_df.copy()
scaled["Age"]/=4
scaled["Release"]/=4
scaled["Rating"]/=10
scaled["Male"]/=10
scaled[["Age","Release","Rating","Male"]]

# 문자라 장르 따로 처리한거 + 다른거 정규화한거 합쳐서 벡터화 끝
# axis=1인 이유 : 열방향으로 바로 옆에 이어붙이기 위해서
movie_vectors=pd.concat([genre,scaled[["Age","Release","Rating","Male"]]], axis=1)


In [10]:
user_vector=[True,False,False,False,False,False,False,False,False,0.5,0.75,0.85,0.5]

# 유클리드 거리 구하는 함수
# np.linalg.norm : 한 벡터의 크기 or 두 벡터 사이의 거리를 구함 (norm)
def euclidean_distance(vector_a,vector_b):
  distance = np.linalg.norm(vector_a - vector_b)
  return distance

### 정렬하는 첫번째 방법 : loc을 사용해서 영화이름으로 직접 돈다
results=[]
for movie_name in movie_vectors.index:
  movie_vector = (movie_vectors.loc[movie_name].to_numpy(dtype=float))
  distance = euclidean_distance(user_vector, movie_vector)
  results.append([movie_name, distance])

distance_df = pd.DataFrame(results, columns=["Movie", "Distance"])
# .reset_index(drop=True) : 원래 인덱스 버리고(drop) 다시 순서대로 번호 매긴다는 뜻
distance_df = distance_df.sort_values(by="Distance", ascending=True).reset_index(drop=True)


### 정렬하는 두번째 방법 : iloc을 사용해서 0번부터 돈다
results=[]
for i in range(len(movie_vectors)):
  movie_vector=movie_vectors.iloc[i].to_numpy(dtype=float)
  distance = np.linalg.norm(movie_vector - user_vector)

  # DataFrame에서 ','로 행과 열 구분해서 선택 (무슨 행의 무슨 열에 있는 값 선택)
  # data.loc["행 선택", "열 선택"]
  results.append([movie_df.loc[i,"Movie"],movie_df.loc[i,"Genre"]
                  , movie_df.loc[i, "Rating"], distance])

distance_df = pd.DataFrame(results, columns=["Movie", "Genre", "Rating", "Distance"])

distance_df = distance_df.sort_values(by="Distance").reset_index(drop=True)


### 정렬하는 세번째 방법 : 브로드캐스팅으로 한번에 싹 계산 (가장 보편)

# for문 없이 전체 영화의 거리를 한번에 계산함
distances = np.linalg.norm(movie_vectors.to_numpy() - user_vector, axis=1)

# 그냥 그대로 거리만 원래꺼에 붙이고 정렬
distance_df = movie_df.copy()
distance_df["Distance"] = distances
distance_df = distance_df.sort_values(by="Distance").reset_index(drop=True)

### 정렬 방법 끝


# 코사인 거리 계산하는 함수
def cosine_distance(vector_a, vector_b):
  # 내적은 np.dot(벡터, 벡터)
  dot_product = np.dot(vector_a, vector_b)
  magnitude = (np.linalg.norm(vector_a) * np.linalg.norm(vector_b))
  cosine_similarity = dot_product/magnitude
  distance = 1 - cosine_similarity
  return distance

# 이번엔 코사인 거리로 정렬
cosine_results=[]
for i in range(len(movie_vectors)) :
  # to_numpy 하는 이유 : 판다스 객체를 그냥 벡터로 바꿔서 계산해야하니까
  movie_vector = movie_vectors.iloc[i].to_numpy(dtype=float)
  distance = cosine_distance(user_vector, movie_vector)
  cosine_results.append([movie_df.loc[i, "Movie"],
                        movie_df.loc[i, "Genre"], movie_df.loc[i, "Rating"], distance])

cosine_df = pd.DataFrame(cosine_results, columns=["Movie", "Genre", "Rating","Cosine Distance"])
cosine_df = cosine_df.sort_values(by="Cosine Distance").reset_index(drop=True)


Action       -1.0
Animation     0.0
Comedy        1.0
Crime         0.0
Drama         0.0
Fantasy       0.0
Historical    0.0
SF            0.0
Thriller      0.0
Age           0.0
Release       0.0
Rating        0.0
Male          0.1
Name: 0, dtype: object
